In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import torchaudio.transforms as T

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set dataset path in Google Drive
DATASET_PATH = '/content/drive/MyDrive/KWS_Project/dataset'


# 1. Custom Dataset Class for Audio Loading

class HeyNovaDataset(Dataset):
    """
    Custom Dataset to load audio files from positive and negative directories,
    pad/crop them to a fixed duration, and extract Mel-Spectrogram features.
    """
    def __init__(self, data_dir, sample_rate=16000, duration=1.5, num_mels=40):
        super().__init__()
        self.data_dir = data_dir
        self.sample_rate = sample_rate
        self.target_length = int(sample_rate * duration)  # 24,000 samples for 1.5 seconds

        # Torchaudio Mel-Spectrogram transform
        self.mel_transform = T.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=512,
            win_length=400,  # 25ms window length
            hop_length=160,  # 10ms hop length
            n_mels=num_mels
        )

        self.file_paths = []
        self.labels = []

        # Load positive samples (Hey Nova -> Label 1)
        pos_dir = os.path.join(data_dir, 'positive')
        if os.path.exists(pos_dir):
            for file in os.listdir(pos_dir):
                if file.endswith('.wav') or file.endswith('.mp3'):
                    self.file_paths.append(os.path.join(pos_dir, file))
                    self.labels.append(1)

        # Load negative samples (Hard Negatives/Noise -> Label 0)
        neg_dir = os.path.join(data_dir, 'negative')
        if os.path.exists(neg_dir):
            for file in os.listdir(neg_dir):
                if file.endswith('.wav') or file.endswith('.mp3'):
                    self.file_paths.append(os.path.join(neg_dir, file))
                    self.labels.append(0)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        label = self.labels[idx]

        # Load audio file
        waveform, sr = torchaudio.load(file_path)

        # Convert stereo to mono if necessary
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Resample waveform if sample rate differs from target
        if sr != self.sample_rate:
            resampler = T.Resample(sr, self.sample_rate)
            waveform = resampler(waveform)

        # Standardize audio length (Padding / Cropping to 1.5 seconds)
        if waveform.shape[1] < self.target_length:
            pad_amount = self.target_length - waveform.shape[1]
            waveform = F.pad(waveform, (0, pad_amount))
        else:
            waveform = waveform[:, :self.target_length]

        # Extract Mel-Spectrogram -> Shape: [40, T]
        mel_spec = self.mel_transform(waveform).squeeze(0)

        return mel_spec, torch.tensor(label, dtype=torch.long)


# 2. Adaptive PCEN Front-End Module (Fixed)

class AdaptivePCEN(nn.Module):
    """
    Adaptive Per-Channel Energy Normalization (APCEN) - Fixed for Numerical Stability
    """
    def __init__(self, num_bands=40, hidden_dim=32, eps=1e-5):
        super().__init__()
        self.num_bands = num_bands
        self.eps = eps

        # Neural feedback controller (GRU + MLP)
        self.controller_gru = nn.GRU(
            input_size=num_bands * 2,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )
        self.controller_fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.ReLU(),
            nn.Linear(32, 2 * num_bands),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Prevent zero inputs
        x = torch.clamp(x, min=1e-5)
        batch_size, num_bands, time_steps = x.size()

        # Step 1: Compute initial smoothed energy M_t
        s_init = 0.025
        M = torch.zeros_like(x)
        M_t = x[:, :, 0]
        M[:, :, 0] = M_t
        for t in range(1, time_steps):
            M_t = (1 - s_init) * M_t + s_init * x[:, :, t]
            M[:, :, t] = M_t

        # Step 2: Prepare neural controller input
        x_shifted = F.pad(x[:, :, :-1], (1, 0))
        controller_input = torch.cat([x, x_shifted], dim=1).transpose(1, 2)

        # Step 3: Predict dynamic parameters
        gru_out, _ = self.controller_gru(controller_input)
        params = self.controller_fc(gru_out)

        alpha_t = params[:, :, :num_bands].transpose(1, 2)
        gamma_raw = params[:, :, num_bands:].transpose(1, 2)

        # Clamp alpha and gamma to safe numerical bounds
        alpha_t = torch.clamp(alpha_t, min=0.01, max=0.99)
        gamma_t = 0.2 + 0.7 * gamma_raw

        # Step 4: Apply SimpPCEN formula with safety bounds
        denom = torch.pow(self.eps + M, alpha_t)
        smooth_x = x / torch.clamp(denom, min=1e-5)
        apcen_out = torch.pow(torch.clamp(smooth_x, min=1e-5), gamma_t)

        return apcen_out


# 3. Complete Hey Nova KWS Model

class HeyNovaKWSModel(nn.Module):
    """
    Full KWS Network consisting of:
    1. Adaptive PCEN front-end for dynamic hospital noise suppression
    2. 2D CNN layers for time-frequency feature extraction
    3. Final classification head (Output classes: 2 -> Negative / Positive)
    """
    def __init__(self, num_mels=40, num_classes=2):
        super().__init__()
        self.apcen = AdaptivePCEN(num_bands=num_mels)

        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2, 2)
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(64, num_classes)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        # Input x shape: [Batch, 40, T]
        x_apcen = self.apcen(x)
        x_img = x_apcen.unsqueeze(1)  # Expand channel dimension: [B, 1, 40, T]

        x = self.pool(F.relu(self.bn1(self.conv1(x_img))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        x = self.global_pool(x).squeeze(-1).squeeze(-1)
        x = self.dropout(x)
        logits = self.fc(x)
        return logits


# 4. Model Training Pipeline

def train_model(dataset_path, epochs=15, batch_size=8, lr=1e-3):
    """
    Main training function to initialize dataset, model, loss function, and optimizer.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Running training on device: {device}")

    # Initialize Dataset and DataLoader
    dataset = HeyNovaDataset(data_dir=dataset_path)
    if len(dataset) == 0:
        print(f"Error: No files found in '{dataset_path}/positive' or '{dataset_path}/negative'!")
        return None

    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Initialize Model, Loss Function, and Optimizer
    model = HeyNovaKWSModel().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Training Loop
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * inputs.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        epoch_loss = total_loss / total
        epoch_acc = (correct / total) * 100
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.2f}%")

    print("\nModel training completed successfully!")
    return model

# Train model using Google Drive path
trained_model = train_model(dataset_path=DATASET_PATH, epochs=15)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running training on device: cpu
Epoch [1/15] - Loss: 0.6954 - Accuracy: 46.53%
Epoch [2/15] - Loss: 0.6424 - Accuracy: 63.37%
Epoch [3/15] - Loss: 0.6070 - Accuracy: 69.31%
Epoch [4/15] - Loss: 0.5795 - Accuracy: 72.28%
Epoch [5/15] - Loss: 0.5222 - Accuracy: 77.23%
Epoch [6/15] - Loss: 0.4428 - Accuracy: 83.17%
Epoch [7/15] - Loss: 0.4666 - Accuracy: 79.21%
Epoch [8/15] - Loss: 0.3624 - Accuracy: 91.09%
Epoch [9/15] - Loss: 0.3073 - Accuracy: 92.08%
Epoch [10/15] - Loss: 0.2540 - Accuracy: 93.07%
Epoch [11/15] - Loss: 0.1893 - Accuracy: 93.07%
Epoch [12/15] - Loss: 0.1939 - Accuracy: 97.03%
Epoch [13/15] - Loss: 0.2379 - Accuracy: 94.06%
Epoch [14/15] - Loss: 0.1358 - Accuracy: 98.02%
Epoch [15/15] - Loss: 0.1683 - Accuracy: 94.06%

Model training completed successfully!


In [ ]:
import librosa

def predict_audio(model, file_path, sample_rate=16000, duration=1.5, num_mels=40):
    model.eval()
    device = next(model.parameters()).device

    # 1. Load Audio with Librosa (Fixes MP3 decoding issues)
    waveform_np, sr = librosa.load(file_path, sr=sample_rate, mono=True)
    waveform = torch.from_numpy(waveform_np).unsqueeze(0)  # Shape: [1, T]

    # 2. Pad or Crop to 1.5s
    target_length = int(sample_rate * duration)
    if waveform.shape[1] < target_length:
        waveform = F.pad(waveform, (0, target_length - waveform.shape[1]))
    else:
        waveform = waveform[:, :target_length]

    # 3. Mel-Spectrogram -> Shape: [1, 40, T]
    mel_transform = T.MelSpectrogram(sample_rate=sample_rate, n_fft=512, win_length=400, hop_length=160, n_mels=num_mels)
    mel_spec = mel_transform(waveform).to(device)

    # 4. Predict
    with torch.no_grad():
        logits = model(mel_spec)
        probs = F.softmax(logits, dim=1)

    neg_prob = probs[0][0].item() * 100
    pos_prob = probs[0][1].item() * 100

    print(f"--- Audio Evaluation: {os.path.basename(file_path)} ---")
    print(f"Confidence 'Hey Nova' (Positive): {pos_prob:.2f}%")
    print(f"Confidence 'Noise/Other' (Negative): {neg_prob:.2f}%")

    if pos_prob > 70:
        print("Result: Wake-word Detected (Hey Nova)")
    else:
        print("Result: Rejected (Not Hey Nova)")

# Test on a sample file directly from Google Drive
test_sample = os.path.join(DATASET_PATH, "negative", "hey_nola_German Narration_03.mp3")
if os.path.exists(test_sample):
    predict_audio(trained_model, test_sample)

--- Audio Evaluation: hey_nola_German Narration_03.mp3 ---
Confidence 'Hey Nova' (Positive): 0.66%
Confidence 'Noise/Other' (Negative): 99.34%
Result: Rejected (Not Hey Nova)


In [ ]:
from google.colab import output
from IPython.display import display, Javascript
import base64

# JavaScript code to capture audio from the browser microphone
RECORD_JS = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader();
  reader.onloadend = () => resolve(reader.result);
  reader.readAsDataURL(blob);
});
var record = async function(time) {
  text = document.createElement('div');
  text.innerHTML = "Recording... Speak now!";
  document.body.appendChild(text);
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  recorder.start();
  await sleep(time);
  recorder.stop();
  text.innerHTML = "Recording finished!";
  await sleep(1000);
  text.remove();
  const blob = new Blob(chunks, { type: 'audio/wav' });
  return await b2text(blob);
};
"""

def record_audio(filename='live_test.wav', seconds=3):
    """
    Triggers the browser microphone to record audio for a specified duration in seconds,
    then saves it locally in WAV format.
    """
    display(Javascript(RECORD_JS))
    s = output.eval_js('record(%d)' % (seconds * 1000))
    b = base64.b64decode(s.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(b)
    print(f"Audio successfully saved: {filename}")

# Record live audio for 3 seconds
record_audio('my_voice.wav', seconds=3)

# Evaluate the recorded live audio using the trained model
predict_audio(trained_model, 'my_voice.wav')

<IPython.core.display.Javascript object>

MessageError: NotAllowedError: Permission denied